In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 50 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 750].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 114
Number of rows left: 114312


In [4]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
20     1002
9      1002
66     1002
80     1002
1      1002
       ... 
40     1002
106    1002
54     1002
102    1002
88     1002
Name: count, Length: 114, dtype: int64
Number of remaining classes in training set: 114
Number of rows in the resampled training set: 114228


In [6]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 500)
    max_depth = trial.suggest_int('max_depth', 10, 100)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 50)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None, 0.2, 0.5, 0.8])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [7]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_dropextremelymore750withSMOTE_HugeExperimental_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=50)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-22 17:31:52,671] A new study created in RDB with name: randomforest_diseases_symptoms_dropextremelymore750withSMOTE_HugeExperimental_study
[I 2025-04-22 17:35:38,102] Trial 0 finished with value: 0.4722485306903378 and parameters: {'n_estimators': 474, 'max_depth': 49, 'min_samples_split': 13, 'min_samples_leaf': 21, 'max_features': 0.8}. Best is trial 0 with value: 0.4722485306903378.


Trial 0: n_estimators=474, max_depth=49, min_samples_split=13, min_samples_leaf=21, max_features=0.8, Accuracy=0.4722


[I 2025-04-22 17:37:12,369] Trial 1 finished with value: 0.4647284951720376 and parameters: {'n_estimators': 168, 'max_depth': 70, 'min_samples_split': 44, 'min_samples_leaf': 27, 'max_features': None}. Best is trial 0 with value: 0.4722485306903378.


Trial 1: n_estimators=168, max_depth=70, min_samples_split=44, min_samples_leaf=27, max_features=None, Accuracy=0.4647


[I 2025-04-22 17:38:07,454] Trial 2 finished with value: 0.4697534800896549 and parameters: {'n_estimators': 425, 'max_depth': 10, 'min_samples_split': 29, 'min_samples_leaf': 6, 'max_features': 'log2'}. Best is trial 0 with value: 0.4722485306903378.


...

Trial 47: n_estimators=318, max_depth=68, min_samples_split=42, min_samples_leaf=12, max_features=sqrt, Accuracy=0.4836


[I 2025-04-22 18:33:12,281] Trial 48 finished with value: 0.47509369375241695 and parameters: {'n_estimators': 421, 'max_depth': 75, 'min_samples_split': 18, 'min_samples_leaf': 27, 'max_features': 0.5}. Best is trial 20 with value: 0.48394441359981333.


Trial 48: n_estimators=421, max_depth=75, min_samples_split=18, min_samples_leaf=27, max_features=0.5, Accuracy=0.4751


[I 2025-04-22 18:34:02,767] Trial 49 finished with value: 0.4823598624049075 and parameters: {'n_estimators': 251, 'max_depth': 59, 'min_samples_split': 50, 'min_samples_leaf': 21, 'max_features': 0.2}. Best is trial 20 with value: 0.48394441359981333.


Trial 49: n_estimators=251, max_depth=59, min_samples_split=50, min_samples_leaf=21, max_features=0.2, Accuracy=0.4824

Best Trial:
FrozenTrial(number=20, state=TrialState.COMPLETE, values=[0.48394441359981333], datetime_start=datetime.datetime(2025, 4, 22, 17, 58, 57, 753760), datetime_complete=datetime.datetime(2025, 4, 22, 17, 59, 11, 568888), params={'n_estimators': 72, 'max_depth': 59, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 'sqrt'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=1), 'max_depth': IntDistribution(high=100, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=50, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=50, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None, 0.2, 0.5, 0.8))}, trial_id=363, value=None)
Best Hyperparameters:
{'n_estimators': 72, 'max_depth': 59, 'min_